# Downloading the Data Pack

In [ ]:
!curl -L https://cloud.uol.de/public.php/dav/files/B849axL35cBZzYD -o metagenomic-read-recruitment-data-pack.tar.gz

: 

# Unpack the Data Pack and Enter the Directory

In [ ]:
!tar -zxvf metagenomic-read-recruitment-data-pack.tar.gz

!cd metagenomic-read-recruitment-data-pack

# Turning our Genome into contigs-db.

In [ ]:
!anvi-gen-contigs-database -f genome.fa -o genome.db

# To Annotate contigs_db Genes with Functions

In [ ]:
!anvi-run-ncbi-cogs -c genome.db --num-threads 4

!anvi-run-hmms -c genome.db

!anvi-run-scg-taxonomy -c genome.db --num-threads 4

# Read Recruitment

# Convert Genome Sequence into a Special File Format for Bowtie2

In [ ]:
!bowtie2-build genome.fa genome

# (1) Perform the Read Recruitment and Store the Results in a SAM File

In [ ]:
!bowtie2 -x genome \
        -1 metagenomes/magdalena-R1.fastq \
        -2 metagenomes/magdalena-R2.fastq \
        -S magdalena.sam

# (2) Convert the SAM File into a BAM File

In [ ]:
!samtools view -F 4 \
              -bS magdalena.sam \
              -o magdalena-RAW.bam

# (3) Sort and Index the BAM File

In [ ]:
!samtools sort magdalena-RAW.bam -o magdalena.bam
!samtools index magdalena.bam

# Profile the Contents of BAM File

In [ ]:
!anvi-profile -i magdalena.bam -c genome.db -o magdalena-profile --cluster

# See the Read recruitment Results in the Anvi’o Interactive Interface using the Program Anvi-Interactive

In [ ]:
! anvi-interactive -c genome.db \
                 -p magdalena-profile/PROFILE.db

# Recruiting Reads from Multiple Samples in a Loop

In [ ]:
!for person in batuhan alejandra jonas jessika
do
    echo "Working on ${person} ..."
    bowtie2 -x genome -1 metagenomes/${person}-R1.fastq -2 metagenomes/${person}-R2.fastq -S ${person}.sam
    samtools view -F 4 -bS ${person}.sam -o ${person}-RAW.bam
    samtools sort ${person}-RAW.bam -o ${person}.bam
    samtools index ${person}.bam
    anvi-profile -i ${person}.bam -c genome.db -o ${person}-profile
    rm -rf ${person}.sam ${person}-RAW.bam
done

# Merge these Individual Profiles into a Final Merged Profile Database

In [ ]:
!anvi-merge *-profile/PROFILE.db \
           -c genome.db \
           -o merged-profiles

# Taking a Look at the Read Recruitment Results

# Does Every Individual Carry a Microbial Population that Matches this Genome?

In [ ]:
!anvi-interactive -p merged-profiles/PROFILE.db \
                 -c genome.db

# Does Every Gene in this Genome Occur in Every Individual?

In [ ]:
!anvi-script-add-default-collection -p merged-profiles/PROFILE.db

In [ ]:
!anvi-interactive -p merged-profiles/PROFILE.db \
                 -c genome.db \
                 -C DEFAULT \
                 -b EVERYTHING \
                 --gene-mode